In [1]:
import requests
import polars as pl
import openfoodfacts


In [2]:
df = pl.scan_parquet("/home/sara/food.parquet") \
       .limit(50_000) \
       .collect()


In [3]:
for col_name, col_type in df.collect_schema().items():
    print(f"{col_name}: {col_type}")

additives_n: Int32
additives_tags: List(String)
allergens_tags: List(String)
brands_tags: List(String)
brands: String
categories: String
categories_tags: List(String)
categories_properties: Struct({'ciqual_food_code': Int32, 'agribalyse_food_code': Int32, 'agribalyse_proxy_food_code': Int32})
checkers_tags: List(String)
ciqual_food_name_tags: List(String)
cities_tags: List(String)
code: String
compared_to_category: String
complete: Int32
completeness: Float32
correctors_tags: List(String)
countries_tags: List(String)
created_t: Int64
creator: String
data_quality_errors_tags: List(String)
data_quality_info_tags: List(String)
data_quality_warnings_tags: List(String)
data_sources_tags: List(String)
ecoscore_data: String
ecoscore_grade: String
ecoscore_score: Int32
ecoscore_tags: List(String)
editors: List(String)
emb_codes_tags: List(String)
emb_codes: String
entry_dates_tags: List(String)
food_groups_tags: List(String)
generic_name: List(Struct({'lang': String, 'text': String}))
images: 

In [16]:
len(df.schema)

111

In [15]:
df.shape

(50000, 111)

In [ ]:
df.head()

In [ ]:
df.describe()


In [ ]:
df.null_count()

In [6]:
null_ratio=df.select(
    ((pl.all().null_count()/df.height)*100).name.suffix("_null_ratio")
)

In [ ]:
print(null_ratio)

In [14]:
high_null = null_ratio.transpose(
    include_header=True,
    header_name="column"
).sort("column_0", descending=True).filter(
    pl.col("column_0") > 80
)

print(high_null)

shape: (27, 2)
┌────────────────────────────┬──────────┐
│ column                     ┆ column_0 │
│ ---                        ┆ ---      │
│ str                        ┆ f64      │
╞════════════════════════════╪══════════╡
│ new_additives_n_null_ratio ┆ 99.972   │
│ owner_fields_null_ratio    ┆ 99.972   │
│ owner_null_ratio           ┆ 99.97    │
│ photographers_null_ratio   ┆ 99.97    │
│ with_sweeteners_null_ratio ┆ 99.352   │
│ …                          ┆ …        │
│ labels_null_ratio          ┆ 81.298   │
│ popularity_tags_null_ratio ┆ 81.128   │
│ scans_n_null_ratio         ┆ 80.98    │
│ unique_scans_n_null_ratio  ┆ 80.98    │
│ labels_tags_null_ratio     ┆ 80.962   │
└────────────────────────────┴──────────┘


In [15]:
high_null.count()

column,column_0
u32,u32
27,27


In [16]:
cols_to_drop = (
    high_null["column"]
    .str.replace("_null_ratio", "") 
    .to_list()
)

In [17]:
df_clean = df.drop(cols_to_drop)


In [18]:
print(f"Before drop: {df.shape[1]} columns")
print(f"After drop:  {df_clean.shape[1]} columns")
print(f"Dropped:     {len(cols_to_drop)} columns")

Before drop: 111 columns
After drop:  84 columns
Dropped:     27 columns


In [ ]:
for i, col in enumerate(df_clean.columns, 1):
    print(f"{i}. {col}")


1. additives_n
2. additives_tags
3. allergens_tags
4. brands_tags
5. brands
6. categories
7. categories_tags
8. categories_properties
9. checkers_tags
10. ciqual_food_name_tags
11. code
12. compared_to_category
13. complete
14. completeness
15. correctors_tags
16. countries_tags
17. created_t
18. creator
19. data_quality_errors_tags
20. data_quality_info_tags
21. data_quality_warnings_tags
22. data_sources_tags
23. ecoscore_data
24. ecoscore_grade
25. ecoscore_tags
26. entry_dates_tags
27. food_groups_tags
28. generic_name
29. images
30. informers_tags
31. ingredients_analysis_tags
32. ingredients_from_palm_oil_n
33. ingredients_n
34. ingredients_original_tags
35. ingredients_percent_analysis
36. ingredients_tags
37. ingredients_text
38. ingredients_with_specified_percent_n
39. ingredients_with_unspecified_percent_n
40. ingredients_without_ciqual_codes_n
41. ingredients_without_ciqual_codes
42. ingredients
43. known_ingredients_n
44. lang
45. languages_tags
46. last_edit_dates_tags
47.

In [31]:
with pl.Config(tbl_cols=-1, tbl_rows=2):
    print(df_clean.head(2))

shape: (2, 84)
┌─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┐
│ add ┆ add ┆ all ┆ bra ┆ bra ┆ cat ┆ cat ┆ cat ┆ che ┆ ciq ┆ cod ┆ com ┆ com ┆ com ┆ cor ┆ cou ┆ cre ┆ cre ┆ dat ┆ dat ┆ dat ┆ dat ┆ eco ┆ eco ┆ eco ┆ ent ┆ foo ┆ gen ┆ ima ┆ inf ┆ ing ┆ ing ┆ ing ┆ ing ┆ ing ┆ ing ┆ ing ┆ ing ┆ ing ┆ ing ┆ ing ┆ ing ┆ kno ┆ lan ┆ lan ┆ las ┆ las ┆ las ┆ las ┆ las ┆ las ┆ mai ┆ max ┆ min ┆ mis ┆ no_ ┆ nov ┆ nov ┆ nov ┆ nuc ┆ nut ┆ nut ┆ nut ┆ nut ┆ nut ┆ obs ┆ pac ┆ pac ┆ pac ┆ pac ┆ pop ┆ pro ┆ pro ┆ pro ┆ qua ┆ rev ┆ ser ┆ ser ┆ sta ┆ tra